<a href="https://colab.research.google.com/github/Muhammad-Waleed-Source/Machine-Learning/blob/main/Day-32/Discretization(Bining).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [53]:
import pandas as pd
import numpy as np

In [54]:
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import KBinsDiscretizer
from sklearn.compose import ColumnTransformer

In [55]:
df = pd.read_csv('train.csv',usecols=['Age','Fare','Survived'])

In [56]:
df.isnull().sum()

,0
Survived,0
Age,177
Fare,0


In [57]:
df.dropna(inplace=True)

In [58]:
df.shape

(714, 3)

In [59]:
df.head()

,Survived,Age,Fare
0,0,22.0,7.2500
1,1,38.0,71.2833
2,1,26.0,7.9250
3,1,35.0,53.1000
4,0,35.0,8.0500


## Train test split

In [60]:
X = df.iloc[:, 1:]
y = df.iloc[:, 0]

In [61]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [62]:
X_test.head()

,Age,Fare
149,42.0,13.00
407,3.0,18.75
53,29.0,26.00
369,24.0,69.30
818,43.0,6.45


In [63]:
y_train.head()

,Survived
328,1
73,0
253,0
719,0
666,0


## Decision Tree

In [64]:
clf = DecisionTreeClassifier()

In [65]:
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

In [66]:
accuracy_score(y_test, y_pred)

0.6363636363636364

In [67]:
np.mean(cross_val_score(DecisionTreeClassifier(), X, y, cv=10, scoring='accuracy'))

np.float64(0.6387128325508608)

## Discretizaion

In [68]:
kbin_age = KBinsDiscretizer(n_bins=15, encode='ordinal', strategy='quantile')
kbin_fare = KBinsDiscretizer(n_bins=15, encode='ordinal', strategy='quantile')

In [69]:
trf = ColumnTransformer([
    ('first', kbin_age, [0]),
    ('second', kbin_fare, [1])
])

In [70]:
X_train_trf = trf.fit_transform(X_train)
X_test_trf = trf.transform(X_test)

In [71]:
trf.named_transformers_

{'first': KBinsDiscretizer(encode='ordinal', n_bins=15),
 'second': KBinsDiscretizer(encode='ordinal', n_bins=15)}

In [72]:
trf.named_transformers_['first']

KBinsDiscretizer(encode='ordinal', n_bins=15)

In [73]:
trf.named_transformers_['first'].n_bins_

array([15])

In [74]:
trf.named_transformers_['first'].bin_edges_

array([array([ 0.42,  6.  , 16.  , 19.  , 21.  , 23.  , 25.  , 28.  , 30.  ,
              32.  , 35.  , 38.  , 42.  , 47.  , 54.  , 80.  ])             ],
      dtype=object)

In [75]:
trf.named_transformers_['second'].n_bins_

array([15])

In [76]:
trf.named_transformers_['second'].bin_edges_

array([array([  0.    ,   7.25  ,   7.775 ,   7.8958,   8.1583,  10.5   ,
               13.    ,  14.4542,  18.75  ,  26.    ,  26.55  ,  31.275 ,
               51.4792,  76.2917, 108.9   , 512.3292])                   ],
      dtype=object)

In [77]:
output = pd.DataFrame({
    'age':X_train['Age'],
    'age_trf':X_train_trf[:,0],
    'fare':X_train['Fare'],
    'fare_trf':X_train_trf[:,1]
})

In [78]:
output['age_labels'] = pd.cut(x=X_train['Age'], bins=trf.named_transformers_['first'].bin_edges_[0].tolist())

output['fare_labels'] = pd.cut(x=X_train['Fare'], bins=trf.named_transformers_['second'].bin_edges_[0].tolist())

In [79]:
output.sample(5)

,age,age_trf,fare,fare_trf,age_labels,fare_labels
299,50.0,13.0,247.5208,14.0,"(47.0, 54.0]","(108.9, 512.329]"
673,31.0,8.0,13.0000,6.0,"(30.0, 32.0]","(10.5, 13.0]"
251,29.0,7.0,10.4625,4.0,"(28.0, 30.0]","(8.158, 10.5]"
393,23.0,5.0,113.2750,14.0,"(21.0, 23.0]","(108.9, 512.329]"
719,33.0,9.0,7.7750,2.0,"(32.0, 35.0]","(7.25, 7.775]"


## Decision Tree after discretization

In [80]:
clf = DecisionTreeClassifier()
clf.fit(X_train_trf,y_train)
y_pred2 = clf.predict(X_test_trf)

In [82]:
accuracy_score(y_test,y_pred2)

# now we can see that our accuracy is improved a bit

0.6363636363636364

In [83]:
X_trf = trf.fit_transform(X)
np.mean(cross_val_score(DecisionTreeClassifier(),X,y,cv=10,scoring='accuracy'))

np.float64(0.6330985915492957)